# Training FourBi với nhiều dataset

Notebook này chuẩn bị dữ liệu và chạy training khi dữ liệu ban đầu có dạng:

```text
root/
  A/{imgs, gt_imgs}
  B/{imgs, gt_imgs}
  ...
```

Code gốc huấn luyện trên patch nhưng validation/test trên ảnh đầy đủ. Vì vậy notebook giữ nguyên `imgs/`, `gt_imgs/` và tạo thêm `imgs_<PATCH_SIZE_RAW>/`, `gt_imgs_<PATCH_SIZE_RAW>/` trong từng dataset. Biến `DATASETS_PATH` là thư mục `root/`, có thể chứa bất kỳ số lượng dataset con nào.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import torch

n_gpu = torch.cuda.device_count()
print(f"Số lượng GPU khả dụng: {n_gpu}")

if n_gpu > 0:
    for i in range(n_gpu):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")

    # Khuyến nghị
    if n_gpu >= 4:
        print("\n✓ Có thể sử dụng 4 GPUs như trong README")
        print("  Lệnh: python -m torch.distributed.launch --nproc_per_node=4 main_task_caption.py ...")
    elif n_gpu > 1:
        print(f"\n⚠ Có {n_gpu} GPUs. Có thể điều chỉnh --nproc_per_node={n_gpu}")
    else:
        print("\n⚠ Chỉ có 1 GPU. Có thể train nhưng sẽ chậm hơn.")
        print("  Cân nhắc giảm batch_size nếu gặp OOM error")
else:
    print("\n✗ Không có GPU. CPU training sẽ rất chậm.")

## 1. Import và cấu hình

Chạy notebook từ thư mục gốc của project (nơi có `train.py`). Sửa `DATASETS_PATH` và các tham số bên dưới trước khi chạy các cell tiếp theo.

In [ ]:
from pathlib import Path
import shlex
import shutil
import subprocess
import sys

import torch

from data.process_image import PatchImage

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'train.py').exists(), 'Hãy mở notebook với working directory là thư mục gốc FourBi.'

# Thư mục chứa A/, B/, ...; mỗi thư mục con có imgs/ và gt_imgs/.
DATASETS_PATH = Path('/workspace/project/final-dataset-binarization/Final_dataset').expanduser()
CHECKPOINT_DIR = PROJECT_ROOT / 'ckpts' 

# PatchImage tạo patch kích thước PATCH_SIZE_RAW; model crop ngẫu nhiên về PATCH_SIZE lúc train.
PATCH_SIZE = 256
OVERLAP_SIZE = PATCH_SIZE //2
PATCH_SIZE_RAW = PATCH_SIZE + 128

# Đặt tên dataset cụ thể, hoặc để None để chọn tự động ở cell phân chia dữ liệu.
EVAL_DATASET_NAME = '2016'
TEST_DATASET_NAME = '2011'

# True sẽ xóa và tạo lại đúng hai thư mục patch được sinh bởi notebook.
REBUILD_PATCHES = False

assert PATCH_SIZE_RAW >= PATCH_SIZE, 'PATCH_SIZE_RAW phải lớn hơn hoặc bằng PATCH_SIZE.'
print('Project:', PROJECT_ROOT)
print('Dataset root:', DATASETS_PATH)

## 2. Tự động tìm và kiểm tra các dataset

Loader đánh giá của project yêu cầu ảnh và ground truth có cùng tên file. Các phần tử không đúng cấu trúc sẽ được báo lỗi sớm thay vì lỗi giữa lúc training.

In [ ]:
SUPPORTED_SUFFIXES = {'.png', '.jpg', '.bmp', '.tif'}
def discover_dataset_paths(root: Path):
    if not root.exists():
        raise FileNotFoundError(f'Không tìm thấy DATASETS_PATH: {root}')
    dataset_paths = sorted(
        path for path in root.iterdir()
        if path.is_dir() and (path / 'imgs').is_dir() and (path / 'gt_imgs').is_dir()
    )
    if not dataset_paths:
        raise ValueError(f'Không tìm thấy dataset con dạng <name>/imgs và <name>/gt_imgs trong {root}')
    return dataset_paths

def validate_dataset(dataset_path: Path):
    imgs_dir = dataset_path / 'imgs'
    gt_dir = dataset_path / 'gt_imgs'

    image_paths = sorted(
        path for path in imgs_dir.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
    )

    if not image_paths:
        raise ValueError(
            f'{dataset_path.name}: imgs/ không có ảnh với đuôi {sorted(SUPPORTED_SUFFIXES)}'
        )

    # Lấy danh sách tên không kèm đuôi của ground truth
    gt_stems = {
        path.stem
        for path in gt_dir.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
    }

    # Chỉ cần cùng tên, khác đuôi vẫn tính là có ground truth
    missing_gt = [
        path.name
        for path in image_paths
        if path.stem not in gt_stems
    ]

    if missing_gt:
        shown = ', '.join(missing_gt[:5])
        raise FileNotFoundError(
            f'{dataset_path.name}: thiếu ground truth cùng tên cho: {shown}'
        )

    return len(image_paths)

dataset_paths = discover_dataset_paths(DATASETS_PATH)
dataset_image_counts = {path.name: validate_dataset(path) for path in dataset_paths}

print(f'Tìm thấy {len(dataset_paths)} dataset:')
for name, count in dataset_image_counts.items():
    print(f'  {name}: {count} ảnh gốc')

## 3. Chọn train / validation / test

`train.py` nhận danh sách toàn bộ dataset rồi loại validation và test khỏi tập train theo tên thư mục. Nếu chỉ có hai dataset, lựa chọn mặc định dùng dataset cuối cho cả validation và test để dataset còn lại vẫn được dùng để train. Với từ ba dataset trở lên, hai dataset cuối được dùng riêng cho validation và test.

In [ ]:
dataset_names = [path.name for path in dataset_paths]
EXCLUDE_TRAIN_DATASET_NAME = '2019'
if len(dataset_names) < 2:
    raise ValueError('Cần ít nhất 2 dataset: một dataset train và một dataset validation/test.')

if TEST_DATASET_NAME is None:
    TEST_DATASET_NAME = dataset_names[-1]
if EVAL_DATASET_NAME is None:
    EVAL_DATASET_NAME = dataset_names[-2] if len(dataset_names) >= 3 else TEST_DATASET_NAME

unknown = {EVAL_DATASET_NAME, TEST_DATASET_NAME}.difference(dataset_names)
if unknown:
    raise ValueError(f'Dataset validation/test không tồn tại: {sorted(unknown)}')

# train_dataset_names = [
#     name for name in dataset_names if name not in {EVAL_DATASET_NAME, TEST_DATASET_NAME}
# ]
train_dataset_names = [
    name for name in dataset_names
    if name not in {EVAL_DATASET_NAME, TEST_DATASET_NAME, EXCLUDE_TRAIN_DATASET_NAME}
]
if not train_dataset_names:
    raise ValueError('Phân chia hiện tại không còn dataset nào để train.')

print('Train      :', train_dataset_names)
print('Validation :', EVAL_DATASET_NAME)
print('Test       :', TEST_DATASET_NAME)

## 4. Sinh patch cho các dataset

Patch được sinh cho tất cả dataset để bạn có thể đổi cách chia train/validation/test mà không phải chuẩn bị lại dữ liệu. Nếu patch đã tồn tại, cell bỏ qua dataset đó. Chỉ bật `REBUILD_PATCHES=True` khi muốn xóa hai thư mục patch tương ứng kích thước hiện tại và tạo lại.

In [ ]:
def patch_dirs(dataset_path: Path):
    return dataset_path / f'imgs_{PATCH_SIZE_RAW}', dataset_path / f'gt_imgs_{PATCH_SIZE_RAW}'

def has_patches(dataset_path: Path):
    img_dir, gt_dir = patch_dirs(dataset_path)
    return img_dir.exists() and gt_dir.exists() and any(img_dir.iterdir()) and any(gt_dir.iterdir())

def create_dataset_patches(dataset_path: Path):
    img_dir, gt_dir = patch_dirs(dataset_path)
    if REBUILD_PATCHES:
        shutil.rmtree(img_dir, ignore_errors=True)
        shutil.rmtree(gt_dir, ignore_errors=True)
    elif has_patches(dataset_path):
        print(f'{dataset_path.name}: đã có patch, bỏ qua.')
        return
    print(f'{dataset_path.name}: đang sinh patch...')
    patcher = PatchImage(
        patch_size=PATCH_SIZE_RAW,
        overlap_size=OVERLAP_SIZE,
        destination_root=str(dataset_path),
    )
    patcher.create_patches(root_original=str(dataset_path))

for dataset_path in dataset_paths:
    create_dataset_patches(dataset_path)

print('\nSố patch đã chuẩn bị:')
for dataset_path in dataset_paths:
    img_dir, gt_dir = patch_dirs(dataset_path)
    num_imgs = len(list(img_dir.glob('*')))
    num_gt = len(list(gt_dir.glob('*')))
    if num_imgs != num_gt or num_imgs == 0:
        raise ValueError(f'{dataset_path.name}: số patch ảnh/ground truth không hợp lệ ({num_imgs}/{num_gt}).')
    print(f'  {dataset_path.name}: {num_imgs} patch')

Sau cell trên, mỗi dataset sẽ có cấu trúc phù hợp với project:

```text
root/A/
  imgs/                 # ảnh đầy đủ cho validation/test
  gt_imgs/              # ground truth đầy đủ
  imgs_384/             # patch train, khi PATCH_SIZE_RAW=384
  gt_imgs_384/          # patch ground truth train
```

## 5. Cấu hình lệnh training

Bạn có thể giảm `BATCH_SIZE`, `NUM_WORKERS` hoặc `EPOCHS` để thử nhanh. Script đã tự chọn CUDA, Apple MPS hoặc CPU theo thiết bị hiện có.

In [ ]:
EXPERIMENT_NAME = f'fourbi_{TEST_DATASET_NAME}'
EPOCHS = 500
BATCH_SIZE = 8
NUM_WORKERS = 4
PATIENCE = 60
LEARNING_RATE = 1.5e-4
LEARNING_RATE_MIN = 1.5e-5
LOAD_DATA_IN_MEMORY = False
USE_WANDB = False
WANDB_PROJECT = 'fourbi'
WANDB_ENTITY = None
RESUME = 'none'  # Hoặc chuỗi định danh có trong tên checkpoint .pth.

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, str(PROJECT_ROOT / 'train.py'),
    '--datasets_paths', *[str(path) for path in dataset_paths],
    '--eval_dataset_name', EVAL_DATASET_NAME,
    '--test_dataset_name', TEST_DATASET_NAME,
    '--experiment_name', EXPERIMENT_NAME,
    '--checkpoint_dir', str(CHECKPOINT_DIR),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--num_workers', str(NUM_WORKERS),
    '--patience', str(PATIENCE),
    '--lr', str(LEARNING_RATE),
    '--lr_min', str(LEARNING_RATE_MIN),
    '--patch_size', str(PATCH_SIZE),
    '--patch_size_raw', str(PATCH_SIZE_RAW),
    '--load_data_in_memory', 'true' if LOAD_DATA_IN_MEMORY else 'false',
    '--resume', RESUME,
]
if USE_WANDB:
    command.extend(['--use_wandb', '--wandb_project', WANDB_PROJECT])
    if WANDB_ENTITY is not None:
        command.extend(['--wandb_entity', WANDB_ENTITY])

device = 'CUDA' if torch.cuda.is_available() else ('Apple MPS' if torch.backends.mps.is_available() else 'CPU')
print('Thiết bị sẽ dùng:', device)
print('Lệnh training:')
print(shlex.join(command))

## 6. Bắt đầu training

Cell này thực sự khởi chạy quá trình huấn luyện. Checkpoint được lưu vào `CHECKPOINT_DIR` sau mỗi epoch và khi PSNR validation tốt hơn (sau các epoch khởi đầu).

In [ ]:
subprocess.run(command, cwd=str(PROJECT_ROOT), check=True)